# Intermediate 05 — Dynamic Authorization & Continuous Access Evaluation

## Scenario

A Claims Agent begins a 30-minute task for Alice.

At start:

```text
task active
risk low
agent active
policy v17
claim v8
approval valid
```

During execution, events can change those facts.

We build a local continuous-authorization engine that can return:

```text
continue
reduce
step_up
pause
revoke
```


In [ ]:
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
import copy, json, time, uuid

def utcnow():
    return datetime.now(timezone.utc)

def iso(dt=None):
    return (dt or utcnow()).isoformat()


## 1 — Session state

In [ ]:
session={
 "id":"session:483",
 "subject":"user:alice",
 "actor":"agent:claims",
 "task":"task:483",
 "resource":"claim:483",
 "grants":{"claim.read","claim.update","payment.create"},
 "status":"active",
 "risk_score":10,
 "policy_version":17,
 "resource_version":8,
 "approval":{
   "id":"apr:1",
   "valid":True,
   "amount":300,
   "resource_version":8
 },
 "lease_until":utcnow()+timedelta(minutes=5),
 "delegation_family":"dlg:483",
 "last_event_time":None,
}
print(json.dumps(session,indent=2,default=str))


## 2 — Decision leases

In [ ]:
def lease_valid(s):
    return s["status"]=="active" and utcnow() < s["lease_until"]

print("lease valid:",lease_valid(session))


## 3 — Dynamic policy

In [ ]:
def evaluate(s,action,amount=0):
    if s["status"]!="active":
        return "revoke","session not active"
    if not lease_valid(s):
        return "step_up","authorization lease expired"
    if s["risk_score"]>=80:
        return "revoke","critical risk"
    if s["risk_score"]>=60:
        return "step_up","high risk"
    if action not in s["grants"]:
        return "deny","action not granted"
    if action=="payment.create":
        a=s["approval"]
        if not a["valid"]:
            return "step_up","approval missing/revoked"
        if amount != a["amount"]:
            return "step_up","parameters changed"
        if s["resource_version"] != a["resource_version"]:
            return "step_up","resource changed after approval"
    return "continue","conditions valid"

print(evaluate(session,"claim.read"))


## 4 — Risk change

In [ ]:
risky=copy.deepcopy(session)
risky["risk_score"]=67
print(evaluate(risky,"claim.update"))

critical=copy.deepcopy(session)
critical["risk_score"]=91
print(evaluate(critical,"claim.update"))


## 5 — Attenuation instead of binary deny

In [ ]:
def apply_risk_policy(s):
    s=copy.deepcopy(s)
    if s["risk_score"]>=80:
        s["status"]="revoked"
        s["grants"]=set()
        return "revoke",s
    if s["risk_score"]>=60:
        s["grants"] &= {"claim.read"}
        return "reduce_to_read_only",s
    return "continue",s

action,reduced=apply_risk_policy(risky)
print(action,reduced["grants"])


## 6 — CAEP-style event envelope

In [ ]:
def make_event(event_type, subject, data=None, when=None):
    return {
      "iss":"https://signals.example",
      "aud":"https://agent-runtime.example",
      "iat":int((when or utcnow()).timestamp()),
      "jti":str(uuid.uuid4()),
      "subject":subject,
      "event_type":event_type,
      "data":data or {}
    }

revocation=make_event(
    "session-revoked",
    {"session_id":"session:483"},
    {"reason":"user disabled"}
)
print(json.dumps(revocation,indent=2))


This is a **training simplification**, not a claim that this exact JSON is the normative CAEP wire representation. In production use the SSF/CAEP specification and libraries/profiles appropriate to your ecosystem.

## 7 — Event processor

In [ ]:
PROCESSED=set()
AUDIT=[]

def process_event(s,event):
    if event["jti"] in PROCESSED:
        return s,"duplicate ignored"
    PROCESSED.add(event["jti"])

    event_time=event["iat"]
    if s["last_event_time"] is not None and event_time < s["last_event_time"]:
        return s,"stale/out-of-order ignored"

    s=copy.deepcopy(s)
    old_status=s["status"]
    et=event["event_type"]

    if et=="session-revoked":
        s["status"]="revoked"
        s["grants"]=set()

    elif et=="token-claims-change":
        removed=set(event["data"].get("remove_grants",[]))
        s["grants"] -= removed

    elif et=="policy-change":
        s["policy_version"]=event["data"]["version"]
        s["lease_until"]=utcnow()  # force fresh decision

    elif et=="risk-change":
        s["risk_score"]=event["data"]["score"]
        _,s=apply_risk_policy(s)

    elif et=="agent-quarantined":
        s["status"]="revoked"
        s["grants"]=set()

    elif et=="approval-revoked":
        s["approval"]["valid"]=False

    elif et=="resource-change":
        s["resource_version"]=event["data"]["version"]

    s["last_event_time"]=event_time
    AUDIT.append({
      "event_id":event["jti"],
      "type":et,
      "old_status":old_status,
      "new_status":s["status"],
      "processed_at":iso(),
    })
    return s,"processed"


## 8 — Session revocation

In [ ]:
s=copy.deepcopy(session)
s,msg=process_event(s,revocation)
print(msg,s["status"],s["grants"])
print(evaluate(s,"claim.read"))


## 9 — Token claims change

In [ ]:
s=copy.deepcopy(session)
event=make_event(
 "token-claims-change",
 {"user_id":"alice"},
 {"remove_grants":["claim.update","payment.create"]}
)
s,_=process_event(s,event)
print(s["grants"])


## 10 — Policy change invalidates lease

In [ ]:
s=copy.deepcopy(session)
event=make_event("policy-change",{},{"version":18})
s,_=process_event(s,event)
print(s["policy_version"],lease_valid(s))
print(evaluate(s,"claim.read"))


## 11 — Refresh/re-evaluate after policy change

In [ ]:
def refresh_lease(s,policy_version,minutes=2):
    s=copy.deepcopy(s)
    if s["status"]!="active":
        raise PermissionError("cannot refresh inactive session")
    s["policy_version"]=policy_version
    s["lease_until"]=utcnow()+timedelta(minutes=minutes)
    return s

s=refresh_lease(s,18)
print(evaluate(s,"claim.read"))


## 12 — Approval TOCTOU

In [ ]:
s=copy.deepcopy(session)
print("before change:",evaluate(s,"payment.create",300))

resource_event=make_event("resource-change",{"claim":"483"},{"version":9})
s,_=process_event(s,resource_event)

print("after resource change:",evaluate(s,"payment.create",300))


## 13 — Parameter-bound approval

In [ ]:
s=copy.deepcopy(session)
print("approved amount:",s["approval"]["amount"])
print("try 300:",evaluate(s,"payment.create",300))
print("try 900:",evaluate(s,"payment.create",900))


## 14 — Approval revocation

In [ ]:
s=copy.deepcopy(session)
e=make_event("approval-revoked",{"approval_id":"apr:1"})
s,_=process_event(s,e)
print(evaluate(s,"payment.create",300))


## 15 — Agent quarantine

In [ ]:
s=copy.deepcopy(session)
e=make_event("agent-quarantined",{"agent_id":"agent:claims"})
s,_=process_event(s,e)
print(s["status"],evaluate(s,"claim.read"))


## 16 — Duplicate event handling

In [ ]:
s=copy.deepcopy(session)
e=make_event("risk-change",{},{"score":70})
s,msg1=process_event(s,e)
s,msg2=process_event(s,e)
print(msg1,msg2)


## 17 — Out-of-order event handling

In [ ]:
s=copy.deepcopy(session)
t=utcnow()
newer=make_event("risk-change",{},{"score":70},t)
older=make_event("risk-change",{},{"score":10},t-timedelta(minutes=2))

s,_=process_event(s,newer)
s,msg=process_event(s,older)
print(msg,"risk remains",s["risk_score"])


## 18 — Decision cache

In [ ]:
DECISION_CACHE={}

def cache_key(s,action):
    return (
      s["subject"],s["actor"],s["task"],s["resource"],action,
      s["policy_version"],s["resource_version"],
      s["risk_score"],s["approval"]["valid"]
    )

def cached_evaluate(s,action,amount=0):
    k=cache_key(s,action)
    item=DECISION_CACHE.get(k)
    if item and utcnow()<item["valid_until"]:
        return item["decision"],"cache"
    d=evaluate(s,action,amount)
    DECISION_CACHE[k]={
      "decision":d,
      "valid_until":min(s["lease_until"],utcnow()+timedelta(seconds=30))
    }
    return d,"fresh"

print(cached_evaluate(session,"claim.read"))
print(cached_evaluate(session,"claim.read"))


## 19 — Cache invalidation

In [ ]:
def invalidate_for_session(session_id):
    # In a production cache maintain indexes from security subjects/session IDs
    # to cache entries. This lab simply clears all.
    DECISION_CACHE.clear()

invalidate_for_session("session:483")
print("cache size",len(DECISION_CACHE))


## 20 — Asynchronous resume

In [ ]:
queued_intent={
 "task":"task:483",
 "action":"claim.update",
 "resource":"claim:483",
 "parameters":{"status":"reviewed"}
}

def resume(intent,current_session):
    # Durable intent does not equal durable authority.
    d=evaluate(current_session,intent["action"])
    if d[0]!="continue":
        raise PermissionError(d)
    return "execute",intent

print(resume(queued_intent,session))


## 21 — Delegation family revocation

In [ ]:
REVOKED_FAMILIES=set()

def revoke_family(family):
    REVOKED_FAMILIES.add(family)

def family_check(s):
    if s["delegation_family"] in REVOKED_FAMILIES:
        return "revoke","delegation family revoked"
    return "continue","family active"

print(family_check(session))
revoke_family("dlg:483")
print(family_check(session))


## 22 — Dynamic RAG authorization

In [ ]:
DOCS=[
 {"id":"doc:483","sensitivity":"internal","text":"Claim 483 note"},
 {"id":"doc:public","sensitivity":"public","text":"Claims FAQ"},
]

def rag_filter(s,docs):
    if s["status"]!="active":
        return []
    if s["risk_score"]>=60:
        return [d for d in docs if d["sensitivity"]=="public"]
    return docs

print([d["id"] for d in rag_filter(session,DOCS)])
print([d["id"] for d in rag_filter(risky,DOCS)])


## 23 — Dynamic tool set

In [ ]:
TOOLS={
 "claim.read":"read",
 "claim.update":"write",
 "payment.create":"high-impact"
}

def available_tools(s):
    if s["status"]!="active":
        return []
    if s["risk_score"]>=80:
        return []
    if s["risk_score"]>=60:
        return [t for t,kind in TOOLS.items() if kind=="read" and t in s["grants"]]
    return [t for t in TOOLS if t in s["grants"]]

print(available_tools(session))
print(available_tools(risky))


## 24 — Claims challenge simulation

In [ ]:
def resource_api(s,action):
    result,reason=evaluate(s,action)
    if result=="continue":
        return {"status":200,"result":"ok"}
    if result=="step_up":
        return {
          "status":401,
          "www_authenticate":{
             "error":"insufficient_authentication",
             "claims":{"fresh_authorization":True},
             "reason":reason
          }
        }
    return {"status":403,"reason":reason}

print(json.dumps(resource_api(risky,"claim.update"),indent=2))


## 25 — Revocation propagation latency

In [ ]:
signal_time=time.time()
time.sleep(0.02)  # simulate transport/processing
enforcement_time=time.time()
latency_ms=(enforcement_time-signal_time)*1000
print(f"simulated revocation propagation: {latency_ms:.2f} ms")


## 26 — Audit

In [ ]:
print(json.dumps(AUDIT[-5:],indent=2))


## 27 — Adversarial tests

In [ ]:
def assert_action(s,action,expected,amount=0):
    actual=evaluate(s,action,amount)[0]
    print(action,actual)
    assert actual==expected

assert_action(session,"claim.read","continue")

x=copy.deepcopy(session)
x["risk_score"]=85
assert_action(x,"claim.update","revoke")

x=copy.deepcopy(session)
x["approval"]["valid"]=False
assert_action(x,"payment.create","step_up",300)

x=copy.deepcopy(session)
x["resource_version"]=9
assert_action(x,"payment.create","step_up",300)


## 28 — Exercise: real SSF/CAEP validation

Replace the simplified lab event with a standards-compliant Security Event Token.

Validate:

```text
issuer
audience
signature
event type
subject identifier
event timestamp
jti
```

Then map standardized CAEP events to your internal authorization state machine.


## 29 — Exercise: event broker

Replace direct `process_event()` calls with:

```text
Kafka / Kinesis / Pub/Sub / EventBridge
```

Test:

- duplicates;
- delayed delivery;
- out-of-order events;
- receiver restart;
- poison messages;
- dead-letter handling.

Measure revocation SLO end-to-end.


## 30 — Exercise: OPA + OpenFGA

Use the included artifacts:

```text
policies/opa/dynamic.rego
policies/openfga/model.fga
```

Design:

```text
OpenFGA -> current relationships
OPA -> current risk/context/policy
```

Final decision:

```text
allow only when relationship and dynamic policy agree
```


## 31 — Exercise: long-running agent

Simulate a 20-step agent workflow.

Inject at random steps:

```text
risk increase
task expiry
policy change
approval revocation
agent quarantine
relationship removal
```

Before every side effect, perform a fresh checkpoint.

Measure how many unsafe actions your control plane prevents.


## 32 — Review questions

1. Why can a valid token contain stale authority?
2. What is a stale authorization window?
3. What are time-, event-, risk-, resource- and context-driven triggers?
4. What is the relationship between SSF and CAEP?
5. What is a Security Event Token?
6. What CAEP event types exist?
7. Why is Microsoft Entra CAE not identical to CAEP?
8. What is a claims challenge?
9. What can step-up mean for an autonomous agent?
10. What is a decision lease?
11. Why is cache-key design security-critical?
12. Which events should invalidate authorization caches?
13. What is TOCTOU?
14. Why bind approval to resource version and parameters?
15. Why must asynchronous agents reacquire authority on resume?
16. How should multi-agent revocation propagate?
17. Why must event handlers be idempotent?
18. How can out-of-order events weaken security?
19. Why combine events with short leases/fresh checks?
20. What should a revocation SLO measure?

# Next course

## Intermediate 06 — Authorization for MCP & Tool Servers
